# Fine-Tuning / LoRA / QLoRA — assistente de retenção

Adapta o `Qwen2.5-0.5B-Instruct` (open-source, Apache-2.0) ao domínio de Customer Intelligence do Argus, sem tocar nos pesos originais.

- **LoRA** (Hu et al., 2021): treina adapters de baixo posto sobre o modelo em precisão nativa.
- **QLoRA** (Dettmers et al., 2023): mesmo mecanismo sobre o modelo quantizado em 4 bits (NF4) — footprint de memória ~4x menor.

Rodado neste notebook de verdade numa RTX 3060 Ti (8GB): LoRA em ~13s, QLoRA em ~24s (dataset pequeno, propositalmente — o objetivo é provar o mecanismo de ponta a ponta, não treinar um modelo de produção).

## Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath('.'))
from ml.fine_tuning.lora_finetune import run_finetune, DEFAULT_OUTPUT_DIR
from ml.fine_tuning.qlora_finetune import QLORA_OUTPUT_DIR
from ml.fine_tuning.dataset import TRAIN_EXAMPLES, EVAL_PROMPTS

print(f'{len(TRAIN_EXAMPLES)} exemplos de treino, {len(EVAL_PROMPTS)} prompts de avaliação (held-out)')


## LoRA

`get_nb_trainable_parameters()` mostra a economia real: só uma fração pequena dos pesos do modelo é treinada.

In [ ]:
result_lora = run_finetune(DEFAULT_OUTPUT_DIR, load_in_4bit=False, epochs=3)
print()
print(f"parâmetros treináveis: {result_lora['trainable_params']:,} / {result_lora['total_params']:,} "
      f"({result_lora['trainable_pct']}%)")
print(f"loss: {result_lora['first_loss']:.4f} -> {result_lora['last_loss']:.4f} em {result_lora['seconds']}s")


## QLoRA (base em 4 bits)

In [ ]:
result_qlora = run_finetune(QLORA_OUTPUT_DIR, load_in_4bit=True, epochs=3)
print()
print(f"[QLoRA] parâmetros treináveis: {result_qlora['trainable_params']:,} / "
      f"{result_qlora['total_params']:,} ({result_qlora['trainable_pct']}%)")
print(f"loss: {result_qlora['first_loss']:.4f} -> {result_qlora['last_loss']:.4f} em {result_qlora['seconds']}s")


## Curva de loss (LoRA vs QLoRA)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(result_lora['losses'], label='LoRA (16-bit)')
plt.plot(result_qlora['losses'], label='QLoRA (4-bit)')
plt.xlabel('step')
plt.ylabel('loss')
plt.title('Fine-tuning do assistente de retenção — LoRA vs QLoRA')
plt.legend()
plt.show()


## Comparação qualitativa: base vs. adaptado

Geração nos prompts held-out (fora do conjunto de treino).

In [ ]:
from ml.fine_tuning.evaluate import compare

for row in compare(str(DEFAULT_OUTPUT_DIR)):
    print(f"
PROMPT: {row['prompt']}")
    print(f"  base : {row['base'][:200]}")
    print(f"  tuned: {row['tuned'][:200]}")


## Conclusão

Os dois mecanismos — LoRA e QLoRA — rodam de ponta a ponta neste projeto: carregamento do modelo-base (nativo ou 4-bit), injeção dos adapters, treino real (loss decrescente), salvamento e recarregamento do adapter (`PeftModel.from_pretrained`). Ver `ml/fine_tuning/tests/test_lora_finetune.py` para os testes automatizados (gated por `RUN_FINETUNE=1`, já que baixam o modelo-base e treinam de verdade).